In [15]:
import os
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd

seasonlist = list(range(1950, 2023))

os.chdir("/import/beegfs/CMIP6/wrf_era5")
geo_em_path = "geo_em.d02.nc"
geo = xr.open_dataset(geo_em_path)

def crop_data (datatocrop , geodata):
    lat_geo = geodata['XLAT_M'].values
    lon_geo = geodata['XLONG_M'].values
    geo_lat_min = lat_geo.min()
    geo_lat_max = lat_geo.max()
    geo_lon_min = lon_geo.min()
    geo_lon_max = lon_geo.max()
    if datatocrop.longitude.max() > 180:
        datatocrop = datatocrop.assign_coords(longitude=(((datatocrop.longitude + 180) % 360) - 180))
        datatocrop = datatocrop.sortby('longitude')
    final_data= datatocrop.where((datatocrop.latitude >= geo_lat_min) & (datatocrop.latitude <= geo_lat_max) &(datatocrop.longitude >= geo_lon_min) & (datatocrop.longitude <= geo_lon_max),drop=True)
    return (final_data)

def calculate_ros_events(data):
    T2 = data['t2m']
    SNOW = data['sd']
    RAIN = data['RAIN']
    print('RAIN calc completed')

    ros_events = (RAIN > 0.254) & (SNOW > 2.54)
    print("ROS Events filtered")

    dates = ros_events['time'].dt.date.data
    hours = ros_events['time'].dt.hour.data
    ros_events = ros_events.assign_coords(Date=('time', dates), Hour=('time', hours))
    
    #saving T2
    T2_avg = T2.mean(dim='time')
    T2_during_ros = T2.where(ros_events)
    T2_ros_avg = T2_during_ros.mean(dim='time')
    print('T2_avg and T2_ros_avg done')
    #saving temps 

    rain_sum = RAIN.sum(dim='time')
    rain_during_ros = RAIN.where(ros_events)
    rain_ros_sum = rain_during_ros.sum(dim='time')

    rain_avg = RAIN.mean(dim='time')
    rain_ros_avg = rain_during_ros.mean(dim='time')

    sf_avg=data['sf'].mean(dim='time')
    sf_during_ros=data['sf'].where(ros_events)
    sf_ros_avg=sf_during_ros.mean(dim='time')

    sf_sum=data['sf'].sum(dim='time')
    sf_sum_ros=sf_during_ros.sum(dim='time')
    
    swe_avg = SNOW.mean(dim='time')
    swe_ros_avg = SNOW.where(ros_events).mean(dim='time')
    
    #sum_snow=SNOW.sum(dim='time')
    #sum_ros_snow=SNOW.where(ros_events).sum(dim='time')

    ros_tally = ros_events.sum(dim='time')
    ros_events_filtered = ros_events.where(ros_events != 0).dropna(dim='time', how='all')
    ros_counts = ros_events_filtered.count(dim='time')
    ros_daily_counts = ros_events_filtered.groupby('Date').count(dim='time')
    ros_days_count = ros_daily_counts.where(ros_daily_counts > 0).count(dim='Date')
    print("ROS Daily counted")
    
    sumtotal_precip=(data['tp']).sum(dim='time')
    sumtotal_precip_ros=data['tp'].where(ros_events).sum(dim='time')
    
    avgtotal_precip=(data['tp']).mean(dim='time')
    avgtotal_precip_ros=data['tp'].where(ros_events).mean(dim='time')
    return {
        'ros_events': ros_events,
        'ros_tally': ros_tally,
        'ros_counts': ros_counts,
        'ros_days_count': ros_days_count,
        'rain_sum': rain_sum,
        'rain_ros_sum': rain_ros_sum,
        'rain_avg': rain_avg,
        'rain_ros_avg': rain_ros_avg,
        'sf_avg':sf_avg,
        'sf_ros_avg':sf_ros_avg,
        'sf_sum': sf_sum,
        'sf_sum_ros':sf_sum_ros,
        'swe_avg': swe_avg,
        'swe_ros_avg': swe_ros_avg,
        #'sum_swe':sum_snow,
        #'sum_ros_swe':sum_ros_snow,
        'T2_avg': T2_avg,
        'T2_ros_avg': T2_ros_avg,
        'sumtotal_precip':sumtotal_precip,
        'sumtotal_precip_ros':sumtotal_precip_ros,
        'avgtotal_precip':avgtotal_precip,
        'avgtotal_precip_ros':avgtotal_precip_ros
    }

def get_season_month_labels(time_index):
    labels, months = [], []
    for t in pd.to_datetime(time_index):
        if t.month in [11, 12]:
            season = f"{t.year}-{t.year + 1}"
        elif t.month in [1, 2, 3]:
            season = f"{t.year - 1}-{t.year}"
        else:
            season = None
        if season:
            labels.append(season)
            months.append(t.month)
        else:
            labels.append(None)
            months.append(None)
    return np.array(labels), np.array(months)

def calculate_vars_by_season_month(full_data):
    time_values = full_data['time'].values
    season_labels, month_labels = get_season_month_labels(time_values)

    month_order = [11, 12, 1, 2, 3]
    unique_pairs = sorted(
        set(zip(season_labels, month_labels)) - {(None, None)},
        key=lambda x: (x[0], month_order.index(x[1]))
    )

    print("Starting ROS + Temp calculations by month within each season...")

    result_vars = {
        'ros_tally': [],
        'ros_counts': [],
        'ros_days_count': [],
        'rain_sum': [],
        'rain_ros_sum': [],
        'rain_avg': [],
        'rain_ros_avg': [],
        'swe_avg': [],
        'swe_ros_avg': [],
        'sf_avg': [],
        'sf_ros_avg': [],
        'sf_sum': [],
        'sf_sum_ros': [],
        'T2_avg': [],
        'T2_ros_avg': [],
        'sumtotal_precip': [],
        'sumtotal_precip_ros': [],
        'avgtotal_precip': [],
        'avgtotal_precip_ros': []
    }

    season_coords = []
    month_coords = []

    for season, month in unique_pairs:
        time_mask = (season_labels == season) & (month_labels == month)
        selected_times = time_values[time_mask]
        monthly_data = full_data.sel(time=selected_times)

        print(f"Processing {season} - Month {month}")
        ros_result = calculate_ros_events(monthly_data)

        for key in result_vars:
            if ros_result[key] is not None:
                result_vars[key].append(ros_result[key])
        
        season_coords.append(season)
        month_coords.append(month)

    combined_ds = xr.Dataset()
    for key in result_vars:
        if result_vars[key]:
            stacked = xr.concat(result_vars[key], dim="stacked_dim")
            stacked = stacked.assign_coords(
                season=("stacked_dim", season_coords),
                month=("stacked_dim", month_coords)
            )
            combined_ds[key] = stacked.set_index(stacked_dim=["season", "month"]).unstack("stacked_dim")

    month_order = [11, 12, 1, 2, 3]
    combined_ds = combined_ds.reindex(month=month_order)

    month_names = {11: "Nov", 12: "Dec", 1: "Jan", 2: "Feb", 3: "Mar"}
    combined_ds = combined_ds.assign_coords(
        month_name=("month", [month_names[int(m)] for m in combined_ds.month.values])
    )

    print("Months in final dataset:", combined_ds.month.values)
    print("Month names:", combined_ds.month_name.values)
    print("Completed ROS monthly calculations by season.")
    return combined_ds

varlist=['t2m','tp','sf','sd']
os.chdir("/import/AKCASC/data/cds/reanalysis-era5-single-levels/")

for start_winter_year in seasonlist :
    year1=start_winter_year
    year2=year1+1
    print(year1,year2)
    data=xr.Dataset()
    months_first_year=['11','12']
    months_second_year=['01','02','03']  
    
    for var in varlist:
        all_months_data=[]
        for month in months_first_year:
            name='reanalysis-era5-single-levels_'
            year1_file=f'_{year1}_{month}.nc'
            #print(f'loading {var} for {year1_file}')
            #ds=xr.open_dataset(var+'/'+name+var+year1_file)
            with xr.open_dataset(var+'/'+name+var+year1_file) as varDS:
                try:
                    varDS = varDS.rename({'valid_time':'time'})  # if the time variable is named valid_time, rename it to time
                    print('changed from valid_time to time')
                except ValueError:
                    pass
            varDS = varDS.drop_vars(['expver', 'number'], errors='ignore')
            da=varDS[var]
            da_cropped=crop_data(da,geo)    
            all_months_data.append(da_cropped)
        
        for month in months_second_year:
            name='reanalysis-era5-single-levels_'
            year2_file=f'_{year2}_{month}.nc'
            #print(f'loading {var} for {year2_file}')
            #ds=xr.open_dataset(var+'/'+name+var+year2_file)
            with xr.open_dataset(var+'/'+name+var+year2_file) as varDS:
                try:
                    varDS = varDS.rename({'valid_time':'time'})  # if the time variable is named valid_time, rename it to time
                except ValueError:
                    pass
            varDS = varDS.drop_vars(['expver', 'number'], errors='ignore')
            da=varDS[var]
            da_cropped=crop_data(da,geo)    
            all_months_data.append(da_cropped)
        data[var]=xr.concat(all_months_data,dim='time',join='override',coords='minimal')
    #Converting ERA5 data to proper units 
    data['sf'] = data['sf'] * 1000
    data['sf'].attrs['sf'] = 'mm'

    data['tp'] = data['tp'] * 1000
    data['tp'].attrs['units'] = 'mm'
    #data = data.drop_vars('tp')

    data['sd']= data['sd']*1000
    data['sd'].attrs['units'] = 'kg/m^-2'

    data['RAIN']=(data['tp']-data['sf'])
    data['RAIN'].attrs['units'] = 'mm'
    data['time']

    #Running ROS calculations 
    ROS=calculate_ros_events(data)
    #seperating months out 
    ros_by_month = calculate_vars_by_season_month(data)
    ros_by_month = ros_by_month.assign_coords(latitude=("latitude", ros_by_month.latitude.values),longitude=("longitude", ros_by_month.longitude.values))

    #Saving the data 
    output_dir = "/center1/DYNDOWN/phutton5/ROS/Raw_ERA5/Raw_ERA5_netcdf"
    os.makedirs(output_dir, exist_ok=True)

    year1 = start_winter_year
    year2 = start_winter_year + 1
    season_label = f"{year1}-{year2}"

    output_filename = f"Raw_ERA5_ROS_Monthly_{season_label}.nc"
    output_path = os.path.join(output_dir, output_filename)

    new_dim_order = ("season", "month", "latitude", "longitude")

    for var in ros_by_month.data_vars:
        ros_by_month[var] = ros_by_month[var].transpose(*new_dim_order)
    
    comp = dict(zlib=True, complevel=4)
    encoding = {var: comp for var in ros_by_month.data_vars}

    ros_by_month.to_netcdf(output_path,format="NETCDF4",encoding=encoding)

1950 1951
changed from valid_time to time
changed from valid_time to time
changed from valid_time to time
changed from valid_time to time
changed from valid_time to time
changed from valid_time to time
RAIN calc completed
ROS Events filtered
T2_avg and T2_ros_avg done
ROS Daily counted
Starting ROS + Temp calculations by month within each season...
Processing 1950-1951 - Month 11
RAIN calc completed
ROS Events filtered
T2_avg and T2_ros_avg done
ROS Daily counted
Processing 1950-1951 - Month 12
RAIN calc completed
ROS Events filtered
T2_avg and T2_ros_avg done
ROS Daily counted
Processing 1950-1951 - Month 1
RAIN calc completed
ROS Events filtered
T2_avg and T2_ros_avg done
ROS Daily counted
Processing 1950-1951 - Month 2
RAIN calc completed
ROS Events filtered
T2_avg and T2_ros_avg done
ROS Daily counted
Processing 1950-1951 - Month 3
RAIN calc completed
ROS Events filtered
T2_avg and T2_ros_avg done
ROS Daily counted
Months in final dataset: [11 12  1  2  3]
Month names: ['Nov' 'Dec'

In [26]:
import os
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd

start_winter_year=1973
number_of_seasons=1

os.chdir("/import/beegfs/CMIP6/wrf_era5")
geo_em_path = "geo_em.d02.nc"
geo = xr.open_dataset(geo_em_path)

def crop_data (datatocrop , geodata):
    lat_geo = geodata['XLAT_M'].values
    lon_geo = geodata['XLONG_M'].values
    geo_lat_min = lat_geo.min()
    geo_lat_max = lat_geo.max()
    geo_lon_min = lon_geo.min()
    geo_lon_max = lon_geo.max()
    if datatocrop.longitude.max() > 180:
        datatocrop = datatocrop.assign_coords(longitude=(((datatocrop.longitude + 180) % 360) - 180))
        datatocrop = datatocrop.sortby('longitude')
    final_data= datatocrop.where((datatocrop.latitude >= geo_lat_min) & (datatocrop.latitude <= geo_lat_max) &(datatocrop.longitude >= geo_lon_min) & (datatocrop.longitude <= geo_lon_max),drop=True)
    return (final_data)

varlist=['t2m','tp','sf','sd']
os.chdir("/import/AKCASC/data/cds/reanalysis-era5-single-levels/")

data=xr.Dataset()

for i in range(number_of_seasons):
    year1=start_winter_year+i
    year2=year1+1
    print(year1,year2)
    
    months_first_year=['11','12']
    months_second_year=['01','02','03']  
    
    for var in varlist:
        all_months_data=[]
        
        for month in months_first_year:
            name='reanalysis-era5-single-levels_'
            year1_file=f'_{year1}_{month}.nc'
            #print(f'loading {var} for {year1_file}')
            #ds=xr.open_dataset(var+'/'+name+var+year1_file)
            with xr.open_dataset(var+'/'+name+var+year1_file) as varDS:
                try:
                    varDS = varDS.rename({'valid_time':'time'})  # if the time variable is named valid_time, rename it to time
                    print('changed from valid_time to time')
                except ValueError:
                    pass
            varDS = varDS.drop_vars(['expver', 'number'], errors='ignore')
            da=varDS[var]
            da_cropped=crop_data(da,geo)    
            all_months_data.append(da_cropped)
        
        for month in months_second_year:
            name='reanalysis-era5-single-levels_'
            year2_file=f'_{year2}_{month}.nc'
            #print(f'loading {var} for {year2_file}')
            #ds=xr.open_dataset(var+'/'+name+var+year2_file)
            with xr.open_dataset(var+'/'+name+var+year2_file) as varDS:
                try:
                    varDS = varDS.rename({'valid_time':'time'})  # if the time variable is named valid_time, rename it to time
                except ValueError:
                    pass
            varDS = varDS.drop_vars(['expver', 'number'], errors='ignore')
            da=varDS[var]
            da_cropped=crop_data(da,geo)    
            all_months_data.append(da_cropped)
        data[var]=xr.concat(all_months_data,dim='time',join='override',coords='minimal')

data['sf'] = data['sf'] * 1000
data['sf'].attrs['sf'] = 'mm'

data['tp'] = data['tp'] * 1000
data['tp'].attrs['units'] = 'mm'
#data = data.drop_vars('tp')

data['sd']= data['sd']*1000
data['sd'].attrs['units'] = 'kg/m^-2'

data['RAIN']=(data['tp']-data['sf'])
data['RAIN'].attrs['units'] = 'mm'


def calculate_ros_events(data):
    T2 = data['t2m']
    SNOW = data['sd']
    RAIN = data['RAIN']
    print('RAIN calc completed')

    ros_events = (RAIN > 0.254) & (SNOW > 2.54)
    print("ROS Events filtered")

    dates = ros_events['time'].dt.date.data
    hours = ros_events['time'].dt.hour.data
    ros_events = ros_events.assign_coords(Date=('time', dates), Hour=('time', hours))
    
    #saving T2
    T2_avg = T2.mean(dim='time')
    T2_during_ros = T2.where(ros_events)
    T2_ros_avg = T2_during_ros.mean(dim='time')
    print('T2_avg and T2_ros_avg done')
    #saving temps 

    rain_sum = RAIN.sum(dim='time')
    rain_during_ros = RAIN.where(ros_events)
    rain_ros_sum = rain_during_ros.sum(dim='time')

    rain_avg = RAIN.mean(dim='time')
    rain_ros_avg = rain_during_ros.mean(dim='time')

    sf_avg=data['sf'].mean(dim='time')
    sf_during_ros=data['sf'].where(ros_events)
    sf_ros_avg=sf_during_ros.mean(dim='time')

    sf_sum=data['sf'].sum(dim='time')
    sf_sum_ros=sf_during_ros.sum(dim='time')
    
    swe_avg = SNOW.mean(dim='time')
    swe_ros_avg = SNOW.where(ros_events).mean(dim='time')
    
    #sum_snow=SNOW.sum(dim='time')
    #sum_ros_snow=SNOW.where(ros_events).sum(dim='time')

    ros_tally = ros_events.sum(dim='time')
    ros_events_filtered = ros_events.where(ros_events != 0).dropna(dim='time', how='all')
    ros_counts = ros_events_filtered.count(dim='time')
    ros_daily_counts = ros_events_filtered.groupby('Date').count(dim='time')
    ros_days_count = ros_daily_counts.where(ros_daily_counts > 0).count(dim='Date')
    print("ROS Daily counted")
    
    sumtotal_precip=(data['tp']).sum(dim='time')
    sumtotal_precip_ros=data['tp'].where(ros_events).sum(dim='time')
    
    avgtotal_precip=(data['tp']).mean(dim='time')
    avgtotal_precip_ros=data['tp'].where(ros_events).mean(dim='time')
    return {
        'ros_events': ros_events,
        'ros_tally': ros_tally,
        'ros_counts': ros_counts,
        'ros_days_count': ros_days_count,
        'rain_sum': rain_sum,
        'rain_ros_sum': rain_ros_sum,
        'rain_avg': rain_avg,
        'rain_ros_avg': rain_ros_avg,
        'sf_avg':sf_avg,
        'sf_ros_avg':sf_ros_avg,
        'sf_sum': sf_sum,
        'sf_sum_ros':sf_sum_ros,
        'swe_avg': swe_avg,
        'swe_ros_avg': swe_ros_avg,
        #'sum_swe':sum_snow,
        #'sum_ros_swe':sum_ros_snow,
        'T2_avg': T2_avg,
        'T2_ros_avg': T2_ros_avg,
        'sumtotal_precip':sumtotal_precip,
        'sumtotal_precip_ros':sumtotal_precip_ros,
        'avgtotal_precip':avgtotal_precip,
        'avgtotal_precip_ros':avgtotal_precip_ros
    }

ROS=calculate_ros_events(data)

def get_season_month_labels(time_index):
    labels, months = [], []
    for t in pd.to_datetime(time_index):
        if t.month in [11, 12]:
            season = f"{t.year}-{t.year + 1}"
        elif t.month in [1, 2, 3]:
            season = f"{t.year - 1}-{t.year}"
        else:
            season = None
        if season:
            labels.append(season)
            months.append(t.month)
        else:
            labels.append(None)
            months.append(None)
    return np.array(labels), np.array(months)

def calculate_vars_by_season_month(full_data):
    time_values = full_data['time'].values
    season_labels, month_labels = get_season_month_labels(time_values)

    month_order = [11, 12, 1, 2, 3]
    unique_pairs = sorted(
        set(zip(season_labels, month_labels)) - {(None, None)},
        key=lambda x: (x[0], month_order.index(x[1]))
    )

    print("Starting ROS + Temp calculations by month within each season...")

    result_vars = {
        'ros_tally': [],
        'ros_counts': [],
        'ros_days_count': [],
        'rain_sum': [],
        'rain_ros_sum': [],
        'rain_avg': [],
        'rain_ros_avg': [],
        'swe_avg': [],
        'swe_ros_avg': [],
        'sf_avg': [],
        'sf_ros_avg': [],
        'sf_sum': [],
        'sf_sum_ros': [],
        'T2_avg': [],
        'T2_ros_avg': [],
        'sumtotal_precip': [],
        'sumtotal_precip_ros': [],
        'avgtotal_precip': [],
        'avgtotal_precip_ros': []
    }

    season_coords = []
    month_coords = []

    for season, month in unique_pairs:
        time_mask = (season_labels == season) & (month_labels == month)
        selected_times = time_values[time_mask]
        monthly_data = full_data.sel(time=selected_times)

        print(f"Processing {season} - Month {month}")
        ros_result = calculate_ros_events(monthly_data)

        for key in result_vars:
            if ros_result[key] is not None:
                result_vars[key].append(ros_result[key])
        
        season_coords.append(season)
        month_coords.append(month)

    combined_ds = xr.Dataset()
    for key in result_vars:
        if result_vars[key]:
            stacked = xr.concat(result_vars[key], dim="stacked_dim")
            stacked = stacked.assign_coords(
                season=("stacked_dim", season_coords),
                month=("stacked_dim", month_coords)
            )
            combined_ds[key] = stacked.set_index(stacked_dim=["season", "month"]).unstack("stacked_dim")

    month_order = [11, 12, 1, 2, 3]
    combined_ds = combined_ds.reindex(month=month_order)

    month_names = {11: "Nov", 12: "Dec", 1: "Jan", 2: "Feb", 3: "Mar"}
    combined_ds = combined_ds.assign_coords(
        month_name=("month", [month_names[int(m)] for m in combined_ds.month.values])
    )

    print("Months in final dataset:", combined_ds.month.values)
    print("Month names:", combined_ds.month_name.values)
    print("Completed ROS monthly calculations by season.")
    return combined_ds

ros_by_month = calculate_vars_by_season_month(data)

ros_by_month = ros_by_month.assign_coords(
    latitude=("latitude", ros_by_month.latitude.values),
    longitude=("longitude", ros_by_month.longitude.values)
)

output_dir = "/center1/DYNDOWN/phutton5/ROS/Raw_ERA5/Raw_ERA5_netcdf"
os.makedirs(output_dir, exist_ok=True)

year1 = start_winter_year
year2 = start_winter_year + number_of_seasons
season_label = f"{year1}-{year2}"

output_filename = f"Raw_ERA5_ROS_Monthly_{season_label}.nc"
output_path = os.path.join(output_dir, output_filename)

new_dim_order = ("season", "month", "latitude", "longitude")

for var in ros_by_month.data_vars:
    ros_by_month[var] = ros_by_month[var].transpose(*new_dim_order)
    
comp = dict(zlib=True, complevel=4)
encoding = {var: comp for var in ros_by_month.data_vars}

ros_by_month.to_netcdf(
    output_path,
    format="NETCDF4",
    encoding=encoding)

1973 1974
RAIN calc completed
ROS Events filtered
T2_avg and T2_ros_avg done
ROS Daily counted
Starting ROS + Temp calculations by month within each season...
Processing 1973-1974 - Month 11
RAIN calc completed
ROS Events filtered
T2_avg and T2_ros_avg done
ROS Daily counted
Processing 1973-1974 - Month 12
RAIN calc completed
ROS Events filtered
T2_avg and T2_ros_avg done
ROS Daily counted
Processing 1973-1974 - Month 1
RAIN calc completed
ROS Events filtered
T2_avg and T2_ros_avg done
ROS Daily counted
Processing 1973-1974 - Month 2
RAIN calc completed
ROS Events filtered
T2_avg and T2_ros_avg done
ROS Daily counted
Processing 1973-1974 - Month 3
RAIN calc completed
ROS Events filtered
T2_avg and T2_ros_avg done
ROS Daily counted
Months in final dataset: [11 12  1  2  3]
Month names: ['Nov' 'Dec' 'Jan' 'Feb' 'Mar']
Completed ROS monthly calculations by season.


In [ ]:
import os
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd

seasonlist = list(range(1950, 2023))

os.chdir("/import/beegfs/CMIP6/wrf_era5")
geo_em_path = "geo_em.d02.nc"
geo = xr.open_dataset(geo_em_path)

def crop_data (datatocrop , geodata):
    lat_geo = geodata['XLAT_M'].values
    lon_geo = geodata['XLONG_M'].values
    geo_lat_min = lat_geo.min()
    geo_lat_max = lat_geo.max()
    geo_lon_min = lon_geo.min()
    geo_lon_max = lon_geo.max()
    if datatocrop.longitude.max() > 180:
        datatocrop = datatocrop.assign_coords(longitude=(((datatocrop.longitude + 180) % 360) - 180))
        datatocrop = datatocrop.sortby('longitude')
    final_data= datatocrop.where((datatocrop.latitude >= geo_lat_min) & (datatocrop.latitude <= geo_lat_max) &(datatocrop.longitude >= geo_lon_min) & (datatocrop.longitude <= geo_lon_max),drop=True)
    return (final_data)

def calculate_ros_events(data):
    T2 = data['t2m']
    SNOW = data['sd']
    RAIN = data['RAIN']
    print('RAIN calc completed')

    ros_events = (RAIN > 0.254) & (SNOW > 2.54)
    print("ROS Events filtered")

    dates = ros_events['time'].dt.date.data
    hours = ros_events['time'].dt.hour.data
    ros_events = ros_events.assign_coords(Date=('time', dates), Hour=('time', hours))
    
    #saving T2
    T2_avg = T2.mean(dim='time')
    T2_during_ros = T2.where(ros_events)
    T2_ros_avg = T2_during_ros.mean(dim='time')
    print('T2_avg and T2_ros_avg done')
    #saving temps 

    rain_sum = RAIN.sum(dim='time')
    rain_during_ros = RAIN.where(ros_events)
    rain_ros_sum = rain_during_ros.sum(dim='time')

    rain_avg = RAIN.mean(dim='time')
    rain_ros_avg = rain_during_ros.mean(dim='time')

    sf_avg=data['sf'].mean(dim='time')
    sf_during_ros=data['sf'].where(ros_events)
    sf_ros_avg=sf_during_ros.mean(dim='time')

    sf_sum=data['sf'].sum(dim='time')
    sf_sum_ros=sf_during_ros.sum(dim='time')
    
    swe_avg = SNOW.mean(dim='time')
    swe_ros_avg = SNOW.where(ros_events).mean(dim='time')
    
    #sum_snow=SNOW.sum(dim='time')
    #sum_ros_snow=SNOW.where(ros_events).sum(dim='time')

    ros_tally = ros_events.sum(dim='time')
    ros_events_filtered = ros_events.where(ros_events != 0).dropna(dim='time', how='all')
    ros_counts = ros_events_filtered.count(dim='time')
    ros_daily_counts = ros_events_filtered.groupby('Date').count(dim='time')
    ros_days_count = ros_daily_counts.where(ros_daily_counts > 0).count(dim='Date')
    print("ROS Daily counted")
    
    sumtotal_precip=(data['tp']).sum(dim='time')
    sumtotal_precip_ros=data['tp'].where(ros_events).sum(dim='time')
    
    avgtotal_precip=(data['tp']).mean(dim='time')
    avgtotal_precip_ros=data['tp'].where(ros_events).mean(dim='time')
    return {
        'ros_events': ros_events,
        'ros_tally': ros_tally,
        'ros_counts': ros_counts,
        'ros_days_count': ros_days_count,
        'rain_sum': rain_sum,
        'rain_ros_sum': rain_ros_sum,
        'rain_avg': rain_avg,
        'rain_ros_avg': rain_ros_avg,
        'sf_avg':sf_avg,
        'sf_ros_avg':sf_ros_avg,
        'sf_sum': sf_sum,
        'sf_sum_ros':sf_sum_ros,
        'swe_avg': swe_avg,
        'swe_ros_avg': swe_ros_avg,
        #'sum_swe':sum_snow,
        #'sum_ros_swe':sum_ros_snow,
        'T2_avg': T2_avg,
        'T2_ros_avg': T2_ros_avg,
        'sumtotal_precip':sumtotal_precip,
        'sumtotal_precip_ros':sumtotal_precip_ros,
        'avgtotal_precip':avgtotal_precip,
        'avgtotal_precip_ros':avgtotal_precip_ros
    }

def get_season_month_labels(time_index):
    labels, months = [], []
    for t in pd.to_datetime(time_index):
        if t.month in [11, 12]:
            season = f"{t.year}-{t.year + 1}"
        elif t.month in [1, 2, 3]:
            season = f"{t.year - 1}-{t.year}"
        else:
            season = None
        if season:
            labels.append(season)
            months.append(t.month)
        else:
            labels.append(None)
            months.append(None)
    return np.array(labels), np.array(months)

def calculate_vars_by_season_month(full_data):
    time_values = full_data['time'].values
    season_labels, month_labels = get_season_month_labels(time_values)

    month_order = [11, 12, 1, 2, 3]
    unique_pairs = sorted(
        set(zip(season_labels, month_labels)) - {(None, None)},
        key=lambda x: (x[0], month_order.index(x[1]))
    )

    print("Starting ROS + Temp calculations by month within each season...")

    result_vars = {
        'ros_tally': [],
        'ros_counts': [],
        'ros_days_count': [],
        'rain_sum': [],
        'rain_ros_sum': [],
        'rain_avg': [],
        'rain_ros_avg': [],
        'swe_avg': [],
        'swe_ros_avg': [],
        'sf_avg': [],
        'sf_ros_avg': [],
        'sf_sum': [],
        'sf_sum_ros': [],
        'T2_avg': [],
        'T2_ros_avg': [],
        'sumtotal_precip': [],
        'sumtotal_precip_ros': [],
        'avgtotal_precip': [],
        'avgtotal_precip_ros': []
    }

    season_coords = []
    month_coords = []

    for season, month in unique_pairs:
        time_mask = (season_labels == season) & (month_labels == month)
        selected_times = time_values[time_mask]
        monthly_data = full_data.sel(time=selected_times)

        print(f"Processing {season} - Month {month}")
        ros_result = calculate_ros_events(monthly_data)

        for key in result_vars:
            if ros_result[key] is not None:
                result_vars[key].append(ros_result[key])
        
        season_coords.append(season)
        month_coords.append(month)

    combined_ds = xr.Dataset()
    for key in result_vars:
        if result_vars[key]:
            stacked = xr.concat(result_vars[key], dim="stacked_dim")
            stacked = stacked.assign_coords(
                season=("stacked_dim", season_coords),
                month=("stacked_dim", month_coords)
            )
            combined_ds[key] = stacked.set_index(stacked_dim=["season", "month"]).unstack("stacked_dim")

    month_order = [11, 12, 1, 2, 3]
    combined_ds = combined_ds.reindex(month=month_order)

    month_names = {11: "Nov", 12: "Dec", 1: "Jan", 2: "Feb", 3: "Mar"}
    combined_ds = combined_ds.assign_coords(
        month_name=("month", [month_names[int(m)] for m in combined_ds.month.values])
    )

    print("Months in final dataset:", combined_ds.month.values)
    print("Month names:", combined_ds.month_name.values)
    print("Completed ROS monthly calculations by season.")
    return combined_ds

varlist=['t2m','tp','sf','sd']
os.chdir("/import/AKCASC/data/cds/reanalysis-era5-single-levels/")

for start_winter_year in seasonlist :
    year1=start_winter_year
    year2=year1+1
    print(year1,year2)
    data=xr.Dataset()
    months_first_year=['11','12']
    months_second_year=['01','02','03']  
    
    for var in varlist:
        all_months_data=[]
        for month in months_first_year:
            name='reanalysis-era5-single-levels_'
            year1_file=f'_{year1}_{month}.nc'
            #print(f'loading {var} for {year1_file}')
            #ds=xr.open_dataset(var+'/'+name+var+year1_file)
            with xr.open_dataset(var+'/'+name+var+year1_file) as varDS:
                try:
                    varDS = varDS.rename({'valid_time':'time'})  # if the time variable is named valid_time, rename it to time
                    print('changed from valid_time to time')
                except ValueError:
                    pass
            varDS = varDS.drop_vars(['expver', 'number'], errors='ignore')
            da=varDS[var]
            da_cropped=crop_data(da,geo)    
            all_months_data.append(da_cropped)
        
        for month in months_second_year:
            name='reanalysis-era5-single-levels_'
            year2_file=f'_{year2}_{month}.nc'
            #print(f'loading {var} for {year2_file}')
            #ds=xr.open_dataset(var+'/'+name+var+year2_file)
            with xr.open_dataset(var+'/'+name+var+year2_file) as varDS:
                try:
                    varDS = varDS.rename({'valid_time':'time'})  # if the time variable is named valid_time, rename it to time
                except ValueError:
                    pass
            varDS = varDS.drop_vars(['expver', 'number'], errors='ignore')
            da=varDS[var]
            da_cropped=crop_data(da,geo)    
            all_months_data.append(da_cropped)
        data[var]=xr.concat(all_months_data,dim='time',join='override',coords='minimal')
    #Converting ERA5 data to proper units 
    data['sf'] = data['sf'] * 1000
    data['sf'].attrs['sf'] = 'mm'

    data['tp'] = data['tp'] * 1000
    data['tp'].attrs['units'] = 'mm'
    #data = data.drop_vars('tp')

    data['sd']= data['sd']*1000
    data['sd'].attrs['units'] = 'kg/m^-2'

    data['RAIN']=(data['tp']-data['sf'])
    data['RAIN'].attrs['units'] = 'mm'
    data['time']

    #Running ROS calculations 
    ROS=calculate_ros_events(data)
    #seperating months out 
    ros_by_month = calculate_vars_by_season_month(data)
    ros_by_month = ros_by_month.assign_coords(latitude=("latitude", ros_by_month.latitude.values),longitude=("longitude", ros_by_month.longitude.values))

    #Saving the data 
    output_dir = "/center1/DYNDOWN/phutton5/ROS/Raw_ERA5/Raw_ERA5_netcdf"
    os.makedirs(output_dir, exist_ok=True)

    year1 = start_winter_year
    year2 = start_winter_year + 1
    season_label = f"{year1}-{year2}"

    output_filename = f"Raw_ERA5_ROS_Monthly_{season_label}.nc"
    output_path = os.path.join(output_dir, output_filename)

    new_dim_order = ("season", "month", "latitude", "longitude")

    for var in ros_by_month.data_vars:
        ros_by_month[var] = ros_by_month[var].transpose(*new_dim_order)
    
    comp = dict(zlib=True, complevel=4)
    encoding = {var: comp for var in ros_by_month.data_vars}

    ros_by_month.to_netcdf(output_path,format="NETCDF4",encoding=encoding)